# Master Analysis Notebook

This notebook provides a complete, end-to-end workflow for the analyses presented in the paper "Architecture Shapes Processing Strategy in Automatic Speech Recognition."

## Section 1: Setup and Configuration

This section handles all necessary imports, defines key configuration variables for the experiments, and sets up the environment.


In [ ]:
# Install necessary libraries
!pip install transformers datasets torch torchaudio scikit-learn pandas tqdm
!pip install nemo_toolkit['asr']


In [ ]:
import os
import gc
import pickle
import torch
import pandas as pd
import analysis_toolkit as a_t

# === Environment Configuration ===
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# === Path Configuration ===
ROOT_DIR = "./" 
DATA_DIR = os.path.join(ROOT_DIR, "data")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
FIG_DIR = os.path.join(ROOT_DIR, "figs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Data will be saved to: {DATA_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")

# === Model Configuration ===
# The full, representative set of models for the paper's analysis.
MODELS_TO_ANALYZE = [
    "openai/whisper-large-v3-turbo",
    "nvidia/canary-1b",
]

# === Dataset Configuration ===
DATASETS_TO_ANALYZE = [
    "PranavBhalerao/l2-arctic-dataset-250",
]

# === Feature Configuration ===
IS_CATEGORICAL_MAP = {
    "gender": True,
    "l1_background": True,
    "duration": False,
    "f0_mean": False,
}


## Section 2: Representation Extraction

This section iterates through the models and datasets defined in the configuration, extracts the hidden layer representations for each, and saves them to disk as pickle files.


In [ ]:
from datasets import load_dataset, Audio

for dataset_name in DATASETS_TO_ANALYZE:
    print(f"--- Processing Dataset: {dataset_name} ---")
    dataset_tag = dataset_name.split("/")[-1]
    
    # Load dataset once
    print("Loading dataset...")
    dataset = load_dataset(dataset_name, split="train")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000, decode=True))

    for model_name in MODELS_TO_ANALYZE:
        print(f"--- Processing Model: {model_name} ---")
        model_tag = model_name.replace("/", "_")
        output_path = os.path.join(DATA_DIR, f"{model_tag}_{dataset_tag}_reps.pkl")
        
        if os.path.exists(output_path):
            print(f"Representations for {model_name} on {dataset_name} already exist. Skipping.")
            continue
        
        try:
            # Load model and processor using the toolkit
            model, processor = a_t.load_model_and_processor(model_name, DEVICE)
            
            # Extract representations using the toolkit
            reps_by_layer, labels = a_t.extract_representations(model, processor, dataset, DEVICE)
            
            # Save the extracted data
            print(f"Saving representations to {output_path}...")
            with open(output_path, "wb") as f:
                pickle.dump({
                    "reps_by_layer": reps_by_layer,
                    "labels": labels
                }, f)
            print("Save complete.")
            
        except Exception as e:
            print(f"Failed to process {model_name} on {dataset_name}. Error: {e}")
        
        # Clean up memory
        if 'model' in locals(): del model
        if 'processor' in locals(): del processor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

print("\n--- Representation extraction complete for all models and datasets. ---")


## Section 3: Core Probing Analysis

This section loads the saved representation files and runs the main linear probing analysis for each model-dataset pair. The results are aggregated into a single DataFrame.


In [ ]:
all_results = []

for dataset_name in DATASETS_TO_ANALYZE:
    dataset_tag = dataset_name.split("/")[-1]
    
    for model_name in MODELS_TO_ANALYZE:
        model_tag = model_name.replace("/", "_")
        data_path = os.path.join(DATA_DIR, f"{model_tag}_{dataset_tag}_reps.pkl")
        results_path = os.path.join(RESULTS_DIR, f"{model_tag}_{dataset_tag}_probing_results.csv")

        if not os.path.exists(data_path):
            print(f"Representation file for {model_name} on {dataset_name} not found. Skipping.")
            continue

        if os.path.exists(results_path):
            print(f"Results for {model_name} on {dataset_name} already exist. Loading from disk.")
            model_results_df = pd.read_csv(results_path)
            all_results.append(model_results_df)
            continue

        try:
            print(f"Loading representations for {model_name} on {dataset_name} from {data_path}...")
            with open(data_path, "rb") as f:
                data = pickle.load(f)
            
            reps_by_layer = data["reps_by_layer"]
            labels = data["labels"]

            # Run probing analysis using the toolkit
            print(f"Running probing analysis for {model_name} on {dataset_name}...")
            model_results_df = a_t.run_probing_analysis(reps_by_layer, labels, IS_CATEGORICAL_MAP)
            model_results_df["model_name"] = model_name
            model_results_df["dataset"] = dataset_name
            
            # Save results
            print(f"Saving probing results to {results_path}...")
            model_results_df.to_csv(results_path, index=False)
            all_results.append(model_results_df)

        except Exception as e:
            print(f"Failed to run probing for {model_name} on {dataset_name}. Error: {e}")

# Aggregate all results
if all_results:
    final_results_df = pd.concat(all_results, ignore_index=True)
    final_output_path = os.path.join(RESULTS_DIR, "all_models_all_datasets_probing_results.csv")
    final_results_df.to_csv(final_output_path, index=False)
    print(f"\n--- All probing results aggregated and saved to {final_output_path} ---")
    display(final_results_df.head())
else:
    print("\nNo results were generated.")
